[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/06_%E5%A3%B2%E4%B8%8A%E3%83%89%E3%83%A9%E3%82%A4%E3%83%90%E3%83%BC%E3%81%AE%E5%88%86%E6%9E%90.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-06. 売上ドライバーの分析（重回帰）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **統計検定2級レベル**（実務でよく使う検定・回帰）。scipy・statsmodels を使います（Colabは導入済み。ローカルは `uv add scipy statsmodels`）。

**舞台設定**：月次の売上は「新規リード数」と「受注数」のどちらに、どれだけ効いているのか。複数の要因で売上を説明する **重回帰** で分解し、施策の優先度づけに使います。

**使う統計（2級）**：重回帰・偏回帰係数・自由度調整済み決定係数・F検定。

### 📋 使うデータ：`monthly_kpi.csv`（36か月の月次KPI）
1行＝1か月。`売上`(万円)を、`新規リード`(件)と`受注数`(件)で説明します。

| 月 | 売上 | 新規リード | 受注数 |
|---|---|---|---|
| 2023-01 | 760 | 68 | 15 |

## 🎯 この章でできるようになること
- 複数の要因で売上を説明する **重回帰** を実行できる
- **偏回帰係数**を「他を一定にしたときの効き目」として読める
- **自由度調整済みR²** と **F検定** でモデルを評価できる

> **前提**：単回帰（実践05）・統計2級（重回帰）　/　**所要**：約30分　/　**レベル**：2級

## 1. データと関係を眺める

In [ ]:
kpi = load('monthly_kpi.csv')
print('相関:')
print(kpi[['売上','新規リード','受注数']].corr().round(2))
kpi.plot(x='新規リード', y='売上', kind='scatter', figsize=(6,4), title='新規リード vs 売上'); plt.show()

## 2. 重回帰：売上を2要因で説明

`売上 ≈ b1×新規リード + b2×受注数 + 切片`。各係数（**偏回帰係数**）は「もう一方を一定にしたまま、その要因が1増えたときの売上の増分」です。

In [ ]:
import statsmodels.formula.api as smf
m = smf.ols('売上 ~ 新規リード + 受注数', data=kpi).fit()
print(m.params.round(2))
print(f'\nリード1件あたり 約{m.params["新規リード"]:.1f}万円 / 受注1件あたり 約{m.params["受注数"]:.1f}万円')

## 3. モデルの当てはまりと有意性

- **R²**：売上のばらつきを何割説明できたか
- **自由度調整済みR²**：意味の薄い変数を入れると下がる（モデル比較はこちら）
- **F検定**：モデル全体が「意味あり」と言えるか

In [ ]:
print(f'R²            = {m.rsquared:.3f}')
print(f'自由度調整済みR² = {m.rsquared_adj:.3f}')
print(f'モデルのF検定 p値 = {m.f_pvalue:.2e}  → 5%で', ('有意' if m.f_pvalue < 0.05 else '有意でない'))

## 4. 多重共線性に注意

`新規リード` と `受注数` はそれ自体強く相関します（リードが増えれば受注も増える）。説明変数どうしが強く相関すると、**個々の係数が不安定**になります（多重共線性）。

In [ ]:
print('説明変数どうしの相関:', round(kpi['新規リード'].corr(kpi['受注数']), 2))
print('→ 0.8前後と高め。係数の値そのものは慎重に解釈する')

## 🧭 まとめ
- **重回帰**は複数要因で売上を分解し、**偏回帰係数**で各要因の効き目を読める。
- モデル評価は **自由度調整済みR²** と **F検定** を使う。
- 説明変数どうしが強く相関すると **多重共線性** で係数が不安定になる。

> ⚠️ **よくある間違い**
> - 素の **R²は変数を増やすと必ず上がる**。モデル比較は自由度調整済みR²で。
> - 偏回帰係数は「他を一定にしたとき」の効果。多重共線性が強いと符号すら不安定になりうる。
> - 回帰はあくまで**関連**。無作為化実験でない限り因果とは言い切れない。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 偏回帰係数が 8.7 のとき、新規リードが10件増えると売上はいくつ増える？ を ans に
ans = None   # 例: 8.7*10
_check('Q1 売上の増分', ans, 8.7*10)

In [ ]:
# Q2: 説明変数を増やすと『必ず上がる』のは 素のR²(=1) か 自由度調整済みR²(=0) か。ans に
ans = None   # ヒント: 罰則の無いほう
_check('Q2 必ず上がるR²', ans, 1)

In [ ]:
# Q3: モデルのF検定 p値=0.001。全体が有意なら1・有意でないなら0 を ans に
ans = None   # ヒント: 0.001 < 0.05 ?
_check('Q3 モデルの有意性', ans, 1)

---
## 🏆 練習問題

**問1.** `売上 ~ 新規リード` の単回帰と、`売上 ~ 新規リード + 受注数` の重回帰で、R²と自由度調整済みR²を比べよう。

**問2.** `受注数 ~ 新規リード` を回帰し、「リードが受注にどれだけつながるか」を読み取ろう。

**問3.**（考察）多重共線性が強いとき、係数の解釈で気をつけることを1〜2行で書こう。

> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/06_%E5%A3%B2%E4%B8%8A%E3%83%89%E3%83%A9%E3%82%A4%E3%83%90%E3%83%BC%E3%81%AE%E5%88%86%E6%9E%90.md)**
>
> 自分で解いてから開きましょう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 重回帰 | 複数の要因で連続値を予測 |
| 偏回帰係数 | 他を一定にしたときの効き目 |
| 自由度調整済みR² | 変数の数で割り引いたR² |
| F検定 | モデル全体が有意かの検定 |
| 多重共線性 | 説明変数どうしが強く相関する問題 |

```python
import statsmodels.formula.api as smf
m = smf.ols('y ~ x1 + x2', data=df).fit()
m.params; m.rsquared_adj; m.f_pvalue
```